# Phase 5 — FREEZE & BUILD THE PIPELINE (prepare once)

The **only** notebook that reads the raw CSV and prepares data. It runs, in a fixed order:

1. Stateless rules (Phase 2) and stateless features (Phase 4), before the split.
2. The 70/15/15 split stratified on q=10 target bins, `random_state=42`. Hashes are asserted against Phase 3.
3. The train-only far-out filter (Phase 2). **Val/test are never filtered.**
4. One `ColumnTransformer` (`FeatureBuilder` + one-hot + locality `TargetEncoder`), **fit on X_train only**
   with target `log1p(y_train)`. Then train, val and test are transformed.
5. Everything is saved to **`prepared_data.joblib`** as **plain numpy arrays and Python builtins only** (no pandas/sklearn
   objects), so the bundle loads under any pandas/sklearn version (e.g. pandas 2.x in Colab, whatever version built it).
   It is then reloaded and verified, including in a fresh process where pandas and sklearn cannot be imported.

Scaling is **not** applied here; it is model-specific and lives in each model notebook's Pipeline.
`y` is saved **in rupees**; `target_transform="log1p"` tells model notebooks to wrap estimators in
`TransformedTargetRegressor(func=np.log1p, inverse_func=np.expm1)`.

Colab: upload `House_Rent_Dataset.csv` into the working directory, then *Runtime → Run all*. Download the
resulting `prepared_data.joblib` for the model notebooks.

In [1]:
import sys, re, json, hashlib, subprocess, time, warnings
import numpy as np
import pandas as pd
import sklearn, joblib, cloudpickle
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, TargetEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
RANDOM_STATE = 42
TARGET = "Rent"
TARGET_TRANSFORM = "log1p"
BUNDLE_PATH = "prepared_data.joblib"
SAVE_SPLIT_CSVS = False          # optional eyeballing copies; the joblib bundle is the source of truth
print("python", sys.version.split()[0], "| sklearn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

python 3.11.9 | sklearn 1.8.0 | pandas 3.0.2 | numpy 2.4.6


## 1. Frozen preparation code (verbatim from Phases 2 and 4; do not edit here)

In [2]:
# ======================= PHASE-2 STATELESS RULES — COPY VERBATIM INTO PHASE 5 =======================
TEXT_COLS = ["Posted On", "Floor", "Area Type", "Area Locality", "City",
             "Furnishing Status", "Tenant Preferred", "Point of Contact"]
FLOOR_LEVEL_MAP = {"Ground": 0, "Upper Basement": -1, "Lower Basement": -2}


def rule_strip_text(df):
    df = df.copy()
    for c in TEXT_COLS:
        df[c] = df[c].astype(str).str.strip()
    df["Area Locality"] = (df["Area Locality"].str.lower()
                           .str.replace(r"[^a-z0-9]+", " ", regex=True)
                           .str.strip())
    return df


def rule_validity(df):
    ok = (df["Rent"] > 0) & (df["Size"] > 0) & (df["BHK"] >= 1) & (df["Bathroom"] >= 1)
    return df[ok]


def rule_dedup_relisted(df):
    return df.drop_duplicates(subset=[c for c in df.columns if c != "Posted On"], keep="first")


def rule_parse_floor(df):
    df = df.copy()
    parts = df["Floor"].str.split(" out of ", n=1, expand=True).reindex(columns=[0, 1])
    cur = parts[0].replace(FLOOR_LEVEL_MAP)
    df["current_floor"] = pd.to_numeric(cur, errors="coerce").astype("float64")
    df["total_floors"] = pd.to_numeric(parts[1], errors="coerce").astype("float64")
    bad = df["current_floor"] > df["total_floors"]          # NaN comparisons are False
    df.loc[bad, "total_floors"] = df.loc[bad, "current_floor"]
    return df


def rule_parse_date(df):
    df = df.copy()
    df["posted_date"] = pd.to_datetime(df["Posted On"], format="%Y-%m-%d", errors="coerce")
    return df


STATELESS_RULES = [rule_strip_text, rule_validity, rule_dedup_relisted, rule_parse_floor, rule_parse_date]


def apply_stateless_rules(df, verbose=True):
    """Apply all distribution-free rules in a fixed order. Returns (clean_df, log_df)."""
    log = []
    for rule in STATELESS_RULES:
        n_before = len(df)
        df = rule(df)
        log.append({"rule": rule.__name__, "rows_before": n_before, "rows_after": len(df),
                    "rows_removed": n_before - len(df)})
    log = pd.DataFrame(log)
    if verbose:
        print(log.to_string(index=False))
    return df.reset_index(drop=True), log
# =====================================================================================================

In [3]:
# ======================= PHASE-2 LEARNED CLEANING — COPY VERBATIM INTO PHASE 5 =======================
class LogIQRClipper(BaseEstimator, TransformerMixin):
    """Clip numeric columns to Tukey fences learned on log1p(x) of the TRAIN data.

    side: "lower", "upper" or "both". k: IQR multiplier. Fences are stored in fit; transform never
    drops rows, it only clips, so val/test can be transformed with the train-learned fences.
    """
    def __init__(self, k=1.5, side="lower"):
        self.k = k
        self.side = side

    def fit(self, X, y=None):
        L = np.log1p(np.asarray(X, dtype="float64"))
        q1, q3 = np.nanpercentile(L, 25, axis=0), np.nanpercentile(L, 75, axis=0)
        iqr = q3 - q1
        self.lower_ = np.expm1(q1 - self.k * iqr) if self.side in ("lower", "both") else np.full(L.shape[1], -np.inf)
        self.upper_ = np.expm1(q3 + self.k * iqr) if self.side in ("upper", "both") else np.full(L.shape[1], np.inf)
        self.n_features_in_ = L.shape[1]
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        A = np.asarray(X, dtype="float64")
        return np.clip(A, self.lower_, self.upper_)

    def get_feature_names_out(self, input_features=None):
        if input_features is not None:
            return np.asarray(input_features, dtype=object)
        return getattr(self, "feature_names_in_", np.asarray([f"x{i}" for i in range(self.n_features_in_)], dtype=object))


def train_only_far_out_mask(X_train, y_train, k=3.0):
    """Keep-mask for TRAIN rows: drop rows whose log(Rent/Size) lies beyond Tukey far-out fences (k=3)
    computed on the train rows themselves. Never call this on val/test."""
    rps = np.log(np.asarray(y_train, dtype="float64") / X_train["Size"].to_numpy(dtype="float64"))
    q1, q3 = np.percentile(rps, [25, 75]); iqr = q3 - q1
    lo, hi = q1 - k * iqr, q3 + k * iqr
    keep = (rps >= lo) & (rps <= hi)
    return keep, {"k": k, "rent_per_sqft_lower": float(np.exp(lo)), "rent_per_sqft_upper": float(np.exp(hi)),
                  "n_dropped": int((~keep).sum())}


NUM_IMPUTER = SimpleImputer(strategy="median")          # fit on X_train in Phase 5
CAT_IMPUTER = SimpleImputer(strategy="most_frequent")   # safety net (0 categorical NaNs today)
# =====================================================================================================

In [4]:
# ============== PHASE-4 STATELESS FEATURES + PHASE-3 SPLIT — COPY VERBATIM INTO PHASE 5 ==============
def add_stateless_features(df):
    df = df.copy()
    df["locality_key"] = df["City"] + " | " + df["Area Locality"]
    df["posted_month"] = df["posted_date"].dt.month.astype("float64")
    df["posted_dow"] = df["posted_date"].dt.dayofweek.astype("float64")
    return df


STRAT_Q = 10
EXPECTED_HASHES = {"train": "82bf32b8437a934e", "val": "55e1909c244731ec", "test": "1874e7d6b55e66dd"}


def index_hash(idx):
    return hashlib.sha256(np.sort(np.asarray(idx)).tobytes()).hexdigest()[:16]


def make_split(df):
    """70/15/15 two-step split stratified on q=10 target bins (bins for splitting ONLY, never a feature)."""
    y = df[TARGET]
    X = df.drop(columns=[TARGET])
    bins = pd.qcut(y, q=STRAT_Q, labels=False, duplicates="drop")
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=bins)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=RANDOM_STATE,
                                                      stratify=bins.loc[X_temp.index])
    got = {"train": index_hash(X_train.index), "val": index_hash(X_val.index), "test": index_hash(X_test.index)}
    assert got == EXPECTED_HASHES, f"split differs from Phase 3: {got}"
    return X_train, X_val, X_test, y_train, y_val, y_test
# =====================================================================================================

In [5]:
# ======================= PHASE-4 FITTED FEATURE BUILDER — COPY VERBATIM INTO PHASE 5 =======================
FEATURE_GROUPS = ("log", "floor", "city_x_size")   # chosen in Phase 4 (see decisions)
LOCALITY_ENCODING = "target"
FB_INPUT_COLS = ["BHK", "Bathroom", "Size", "current_floor", "total_floors", "posted_date",
                 "posted_month", "posted_dow", "City"]


class FeatureBuilder(BaseEstimator, TransformerMixin):
    """Numeric feature construction whose parameters are learned on the fit data (train / CV fold-train only)."""
    def __init__(self, groups=FEATURE_GROUPS, clip_k=1.5):
        self.groups = groups
        self.clip_k = clip_k

    def fit(self, X, y=None):
        X = pd.DataFrame(X, columns=FB_INPUT_COLS) if not hasattr(X, "columns") else X
        self.clipper_ = LogIQRClipper(k=self.clip_k, side="lower").fit(X[["Size"]])
        self.floor_imputer_ = SimpleImputer(strategy="median").fit(X[["current_floor", "total_floors"]])
        self.start_date_ = pd.to_datetime(X["posted_date"]).min()
        self.cities_ = sorted(X["City"].unique())
        self.feature_names_out_ = np.asarray(list(self._build(X.head(2)).columns), dtype=object)
        self.n_features_in_ = X.shape[1]
        return self

    def _build(self, X):
        g = set(self.groups)
        out = pd.DataFrame(index=X.index)
        size = self.clipper_.transform(X[["Size"]]).ravel()
        bhk, bath = X["BHK"].to_numpy("float64"), X["Bathroom"].to_numpy("float64")
        out["BHK"], out["Bathroom"] = bhk, bath
        if "size_raw" in g:
            out["size"] = size
        if "log" in g:
            out["log_size"], out["log_bhk"], out["log_bath"] = np.log1p(size), np.log1p(bhk), np.log1p(bath)
        if "floor" in g:
            fl = self.floor_imputer_.transform(X[["current_floor", "total_floors"]])
            cur, tot = fl[:, 0], np.maximum(fl[:, 1], fl[:, 0])
            out["current_floor"], out["total_floors"] = cur, tot
            out["floor_ratio"] = np.clip(cur, 0, None) / np.maximum(tot, 1)
            out["is_basement"] = (cur < 0).astype("float64")
            out["log_total_floors"] = np.log1p(tot)
        if "ratios" in g:
            out["size_per_bhk"] = size / bhk
            out["log_size_per_bhk"] = np.log1p(size / bhk)
            out["bath_per_bhk"] = bath / bhk
        if "date" in g:
            out["days_since_start"] = (pd.to_datetime(X["posted_date"]) - self.start_date_).dt.days.to_numpy("float64")
            out["posted_month"] = X["posted_month"].to_numpy("float64")
        if "dow" in g:
            out["posted_dow"] = X["posted_dow"].to_numpy("float64")
        if "city_x_size" in g:
            ls = np.log1p(size)
            for c in self.cities_:
                out[f"logsize_x_{c}"] = ls * (X["City"].to_numpy() == c)
        return out

    def transform(self, X):
        X = pd.DataFrame(X, columns=FB_INPUT_COLS) if not hasattr(X, "columns") else X
        return self._build(X).to_numpy("float64")

    def get_feature_names_out(self, input_features=None):
        return self.feature_names_out_


CAT_COLS = ["City", "Furnishing Status", "Area Type", "Tenant Preferred", "Point of Contact"]


def build_preprocessor(groups=FEATURE_GROUPS, locality=LOCALITY_ENCODING):
    """Unfitted ColumnTransformer. locality ∈ {"target", "count", None}. Fit on X_train ONLY."""
    parts = [("num", FeatureBuilder(groups=groups), FB_INPUT_COLS),
             ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10,
                                   sparse_output=False), CAT_COLS)]
    if locality == "target":
        # cross-fitted inside fit_transform (no row sees its own target); smoothing learned ("auto")
        parts.append(("loc", TargetEncoder(target_type="continuous", smooth="auto", cv=5,
                                           shuffle=True, random_state=RANDOM_STATE), ["locality_key"]))
    elif locality == "count":
        parts.append(("loc", LocalityCountEncoder(), ["locality_key"]))
    return ColumnTransformer(parts, remainder="drop", verbose_feature_names_out=True)


class LocalityCountEncoder(BaseEstimator, TransformerMixin):
    """log1p(number of TRAIN rows sharing the locality); unseen → 0. Uses no target."""
    def fit(self, X, y=None):
        s = pd.Series(np.asarray(X).ravel())
        self.counts_ = s.value_counts().to_dict()
        self.n_features_in_ = 1
        return self

    def transform(self, X):
        s = pd.Series(np.asarray(X).ravel())
        return np.log1p(s.map(self.counts_).fillna(0).to_numpy("float64")).reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(["locality_count_log"], dtype=object)
# =====================================================================================================

## 2. Stateless rules + features → split → train-only far-out filter

In [6]:
raw = pd.read_csv("House_Rent_Dataset.csv")
assert raw.shape == (4746, 12), raw.shape
clean, rule_log = apply_stateless_rules(raw)
clean = add_stateless_features(clean)

X_train_raw, X_val_raw, X_test_raw, y_train_raw, y_val, y_test = make_split(clean)   # asserts Phase-3 hashes
print(f"\nSplit OK: train {len(X_train_raw)} | val {len(X_val_raw)} | test {len(X_test_raw)}")

keep, far_info = train_only_far_out_mask(X_train_raw, y_train_raw, k=3.0)
dropped_train_ids = X_train_raw.index[~keep].tolist()
X_train_raw, y_train = X_train_raw.loc[keep], y_train_raw.loc[keep]
print("Train-only far-out filter:", far_info)
display(clean.loc[dropped_train_ids, ["BHK", "Bathroom", "Size", "City", "Area Locality", TARGET]])
print("Final rows: train", len(X_train_raw), "| val", len(X_val_raw), "| test", len(X_test_raw))

               rule  rows_before  rows_after  rows_removed
    rule_strip_text         4746        4746             0
      rule_validity         4746        4746             0
rule_dedup_relisted         4746        4737             9
   rule_parse_floor         4737        4737             0
    rule_parse_date         4737        4737             0

Split OK: train 3315 | val 711 | test 711
Train-only far-out filter: {'k': 3.0, 'rent_per_sqft_lower': 0.4369066666666667, 'rent_per_sqft_upper': 1271.5657552083335, 'n_dropped': 2}


,BHK,Bathroom,Size,City,Area Locality,Rent
4644,3,3,10,Hyderabad,ghatkesar nh 2 2,15000
1833,3,3,2500,Bangalore,marathahalli,3500000


Final rows: train 3313 | val 711 | test 711


## 3. Fit the preprocessor on X_train ONLY, then transform all splits

In [7]:
preprocessor = build_preprocessor()                 # FEATURE_GROUPS + LOCALITY_ENCODING frozen in Phase 4
t0 = time.perf_counter()
# fit_transform on train → the locality TargetEncoder is cross-fitted (no train row sees its own target)
Z_train = preprocessor.fit_transform(X_train_raw, np.log1p(y_train))
Z_val = preprocessor.transform(X_val_raw)           # full-train encoder; no val/test information used
Z_test = preprocessor.transform(X_test_raw)
print(f"Fitted in {time.perf_counter() - t0:.2f}s on {len(X_train_raw)} train rows only")

def clean_name(n):
    n = n.replace("loc__locality_key", "locality_target_enc").replace("num__", "").replace("cat__", "")
    return re.sub(r"[^0-9A-Za-z_]+", "_", n).strip("_")

raw_names = list(preprocessor.get_feature_names_out())
feature_names = [clean_name(n) for n in raw_names]
assert len(set(feature_names)) == len(feature_names), "feature names must be unique"
display(pd.DataFrame({"saved_name": feature_names, "sklearn_name": raw_names}))

def to_frame(Z, idx):
    return pd.DataFrame(np.asarray(Z, dtype="float64"), columns=feature_names, index=idx)

X_train = to_frame(Z_train, X_train_raw.index)
X_val = to_frame(Z_val, X_val_raw.index)
X_test = to_frame(Z_test, X_test_raw.index)
y_train, y_val, y_test = (s.astype("int64").rename(TARGET) for s in (y_train, y_val, y_test))

Fitted in 0.05s on 3313 train rows only


,saved_name,sklearn_name
0,BHK,num__BHK
1,Bathroom,num__Bathroom
2,log_size,num__log_size
3,log_bhk,num__log_bhk
4,log_bath,num__log_bath
5,current_floor,num__current_floor
6,total_floors,num__total_floors
7,floor_ratio,num__floor_ratio
8,is_basement,num__is_basement
9,log_total_floors,num__log_total_floors


## 4. Integrity checks

In [8]:
checks = {
    "no NaN/inf in X": all(np.isfinite(x.to_numpy()).all() for x in (X_train, X_val, X_test)),
    "identical column order": list(X_train.columns) == list(X_val.columns) == list(X_test.columns) == feature_names,
    "all X float64": all((x.dtypes == "float64").all() for x in (X_train, X_val, X_test)),
    "all y int64 rupees": all(s.dtype == "int64" and s.min() > 0 for s in (y_train, y_val, y_test)),
    "X/y indices aligned": all(x.index.equals(s.index) for x, s in ((X_train, y_train), (X_val, y_val), (X_test, y_test))),
    "no overlap between splits": not (set(X_train.index) & set(X_val.index) or set(X_train.index) & set(X_test.index)
                                      or set(X_val.index) & set(X_test.index)),
    "no Rent / bin / raw string columns in X": not any(re.fullmatch(r"rent|bins?|strat.*|posted_on|posted_date|floor|area_locality|locality_key", c, re.I)
                                                        for c in feature_names),
    "val/test not filtered (sizes 711/711)": (len(X_val), len(X_test)) == (711, 711),
    "far-out rows dropped only from train": len(dropped_train_ids) == far_info["n_dropped"]
                                             and not set(dropped_train_ids) & (set(X_val.index) | set(X_test.index)),
}
for k, v in checks.items():
    print(f"[{'OK' if v else 'FAIL'}] {k}")
assert all(checks.values()), "integrity check failed — STOP"
print(f"\nShapes: X_train {X_train.shape} | X_val {X_val.shape} | X_test {X_test.shape}")

[OK] no NaN/inf in X
[OK] identical column order
[OK] all X float64
[OK] all y int64 rupees
[OK] X/y indices aligned
[OK] no overlap between splits
[OK] no Rent / bin / raw string columns in X
[OK] val/test not filtered (sizes 711/711)
[OK] far-out rows dropped only from train

Shapes: X_train (3313, 34) | X_val (711, 34) | X_test (711, 34)


## 5. Fingerprint
Two layers:
* **`reference`** (platform-independent: row counts, ordered feature names, split-index hashes, target hashes).
  Model notebooks hard-code `EXPECTED_REFERENCE_ID` printed below and STOP on mismatch.
* **`content`** (hashes of the exact float64 matrices). Verified against the bundle's own record, so it catches any
  corruption or edit of the saved data.

In [9]:
def _h(b):
    return hashlib.sha256(b).hexdigest()[:16]

def compute_fingerprint(X_train, X_val, X_test, y_train, y_val, y_test, feature_names):
    splits = {"train": (X_train, y_train), "val": (X_val, y_val), "test": (X_test, y_test)}
    ref = {"n_rows": {k: int(len(x)) for k, (x, _) in splits.items()},
           "n_features": len(feature_names),
           "feature_names_hash": _h("|".join(feature_names).encode()),
           "index_hash_sorted": {k: _h(np.sort(x.index.to_numpy("int64")).tobytes()) for k, (x, _) in splits.items()},
           "index_hash_ordered": {k: _h(x.index.to_numpy("int64").tobytes()) for k, (x, _) in splits.items()},
           "target_hash": {k: _h(y.to_numpy("int64").tobytes()) for k, (_, y) in splits.items()}}
    ref["reference_id"] = _h(json.dumps(ref, sort_keys=True).encode())
    content = {k: _h(np.ascontiguousarray(x.to_numpy("float64")).tobytes()) for k, (x, _) in splits.items()}
    return {"reference": ref, "content": content}

fingerprint = compute_fingerprint(X_train, X_val, X_test, y_train, y_val, y_test, feature_names)
fingerprint["phase3_prefilter_split_hashes"] = EXPECTED_HASHES
print(json.dumps(fingerprint, indent=2))
print("\n>>> EXPECTED_REFERENCE_ID for model notebooks:", fingerprint["reference"]["reference_id"])

{
  "reference": {
    "n_rows": {
      "train": 3313,
      "val": 711,
      "test": 711
    },
    "n_features": 34,
    "feature_names_hash": "aa99ef686f8568a9",
    "index_hash_sorted": {
      "train": "a867a4b143e4e736",
      "val": "55e1909c244731ec",
      "test": "1874e7d6b55e66dd"
    },
    "index_hash_ordered": {
      "train": "70c508616a6835c0",
      "val": "15c95c3f2d009ca2",
      "test": "09e7a8fc85bbad8b"
    },
    "target_hash": {
      "train": "653375dfabe89af0",
      "val": "da5337c1fac3f7ee",
      "test": "a675e4ec80eeab9a"
    },
    "reference_id": "70560d1bd70af7cd"
  },
  "content": {
    "train": "45607f3d45155c30",
    "val": "9abca1d0c8a63c34",
    "test": "ccff150b1ed3fc71"
  },
  "phase3_prefilter_split_hashes": {
    "train": "82bf32b8437a934e",
    "val": "55e1909c244731ec",
    "test": "1874e7d6b55e66dd"
  }
}

>>> EXPECTED_REFERENCE_ID for model notebooks: 70560d1bd70af7cd


## 6. Save the bundle

In [10]:
# Version-independent storage: numpy arrays + Python builtins only (pandas pickles are NOT portable across
# pandas 2 <-> 3). Model notebooks rebuild DataFrames with `frames_from_bundle(d)`; values/dtypes are identical.
bundle = {
    "X_train": np.ascontiguousarray(X_train.to_numpy("float64")),
    "X_val": np.ascontiguousarray(X_val.to_numpy("float64")),
    "X_test": np.ascontiguousarray(X_test.to_numpy("float64")),
    "y_train": y_train.to_numpy("int64"), "y_val": y_val.to_numpy("int64"), "y_test": y_test.to_numpy("int64"),
    "index_train": X_train.index.to_numpy("int64"), "index_val": X_val.index.to_numpy("int64"),
    "index_test": X_test.index.to_numpy("int64"),
    "feature_names": [str(n) for n in feature_names],
    "storage_format": "numpy-v1: X_* float64 2-D arrays, y_* int64 rupees, index_* int64 row ids; rebuild with frames_from_bundle(d)",
    "fingerprint": fingerprint,
    "target_transform": TARGET_TRANSFORM,           # "log1p" → wrap models in TransformedTargetRegressor(log1p, expm1)
    # fitted ColumnTransformer, cloudpickled so the bundle loads without the custom class definitions
    "preprocessor": cloudpickle.dumps(preprocessor),
    "preprocessor_format": "cloudpickle-bytes (restore: cloudpickle.loads(d['preprocessor']))",
    "meta": {
        "random_state": RANDOM_STATE, "split": "70/15/15 two-step (0.15, then 0.1765), stratified on qcut(y, q=10)",
        "strat_q": STRAT_Q, "feature_groups": list(FEATURE_GROUPS), "locality_encoding": LOCALITY_ENCODING,
        "rule_log": json.loads(rule_log.to_json(orient="records")), "far_out_filter": json.loads(json.dumps(far_info)),
        "train_rows_dropped_far_out": [int(i) for i in dropped_train_ids],
        "rows": {"raw": int(len(raw)), "after_rules": int(len(clean))},
        "notes": ["X is encoded but UNSCALED; scaling is model-specific (inside each model Pipeline, fit on X_train).",
                  "X_train locality_target_enc is cross-fitted (sklearn TargetEncoder.fit_transform); val/test use the full-train encoder.",
                  "CV on these saved matrices does not re-fit the locality encoder per fold; Phase 4 measured the effect at <=0.5% RMSE.",
                  "The cloudpickled preprocessor is reference-only and needs the library versions in meta.versions to restore; model notebooks never need it."],
        "versions": {"python": sys.version.split()[0], "sklearn": sklearn.__version__, "pandas": pd.__version__,
                     "numpy": np.__version__, "cloudpickle": cloudpickle.__version__},
    },
}
ALLOWED = (dict, list, tuple, str, int, float, bool, type(None), bytes)
def assert_portable(obj, path="bundle"):
    if isinstance(obj, np.ndarray):
        assert obj.dtype.kind in "fiub", f"{path}: ndarray dtype {obj.dtype} not portable"
    elif isinstance(obj, dict):
        for k, v in obj.items():
            assert type(k) is str, f"{path}: non-str key {k!r}"
            assert_portable(v, f"{path}.{k}")
    elif isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj):
            assert_portable(v, f"{path}[{i}]")
    else:
        assert type(obj) in ALLOWED, f"{path}: type {type(obj).__name__} is not version-independent"
assert_portable(bundle)
print("Portability check: bundle holds only numpy arrays + Python builtins.")
joblib.dump(bundle, BUNDLE_PATH, compress=3)
import os
print(f"Saved {BUNDLE_PATH} ({os.path.getsize(BUNDLE_PATH) / 1024:.0f} KB)")
if SAVE_SPLIT_CSVS:
    for k in ("train", "val", "test"):
        pd.DataFrame(bundle[f"X_{k}"], columns=feature_names, index=bundle[f"index_{k}"]).assign(
            Rent=bundle[f"y_{k}"]).to_csv(f"prepared_{k}.csv")

Portability check: bundle holds only numpy arrays + Python builtins.
Saved prepared_data.joblib (189 KB)


## 7. Verify: reload and compare, then load in a **fresh Python process** (as a model notebook would)
`frames_from_bundle` + `verify_bundle` below are the reference code the model notebooks reuse.

In [11]:
def frames_from_bundle(d):
    # Rebuild DataFrames/Series (float64 X, int64 y in rupees, original row ids as index).
    out = {}
    for s in ("train", "val", "test"):
        idx = pd.Index(d[f"index_{s}"], dtype="int64")
        out[f"X_{s}"] = pd.DataFrame(d[f"X_{s}"], columns=list(d["feature_names"]), index=idx)
        out[f"y_{s}"] = pd.Series(d[f"y_{s}"], index=idx, name="Rent")
    return out

def verify_bundle(d, expected_reference_id):
    f = frames_from_bundle(d)
    fp = compute_fingerprint(f["X_train"], f["X_val"], f["X_test"], f["y_train"], f["y_val"], f["y_test"], d["feature_names"])
    problems = []
    if fp["reference"]["reference_id"] != expected_reference_id:
        problems.append(f"reference_id {fp['reference']['reference_id']} != expected {expected_reference_id}")
    if fp["reference"] != d["fingerprint"]["reference"]:
        problems.append("reference fingerprint differs from the one stored in the bundle")
    if fp["content"] != d["fingerprint"]["content"]:
        problems.append("X content hash differs from the one stored in the bundle")
    if any(d[f"X_{s}"].shape[1] != len(d["feature_names"]) for s in ("train", "val", "test")):
        problems.append("X width != len(feature_names)")
    if problems:
        raise RuntimeError("FINGERPRINT MISMATCH — STOP:\n  " + "\n  ".join(problems))
    return fp["reference"]

EXPECTED_REFERENCE_ID = fingerprint["reference"]["reference_id"]
d = joblib.load(BUNDLE_PATH)
ref = verify_bundle(d, EXPECTED_REFERENCE_ID)
f = frames_from_bundle(d)
for s in ("train", "val", "test"):
    pd.testing.assert_frame_equal(f[f"X_{s}"], {"train": X_train, "val": X_val, "test": X_test}[s], check_column_type=False)
    pd.testing.assert_series_equal(f[f"y_{s}"], {"train": y_train, "val": y_val, "test": y_test}[s])
print("In-process reload: fingerprint OK; rebuilt frames identical to the prepared ones (values, dtypes, index, names).")

pre2 = cloudpickle.loads(d["preprocessor"])
assert np.array_equal(pre2.transform(X_val_raw), X_val.to_numpy()), "restored preprocessor must reproduce X_val"
assert np.array_equal(pre2.transform(X_test_raw), X_test.to_numpy()), "restored preprocessor must reproduce X_test"
print("Restored preprocessor reproduces X_val and X_test exactly.")

In-process reload: fingerprint OK; rebuilt frames identical to the prepared ones (values, dtypes, index, names).
Restored preprocessor reproduces X_val and X_test exactly.


In [12]:
# Fresh process where pandas and sklearn CANNOT be imported: proves the bundle has no version-bound objects
# (this is what makes it load under Colab's pandas 2.x regardless of the pandas that built it).
probe = f'''
import sys
sys.modules["pandas"] = None; sys.modules["sklearn"] = None      # any attempt to import them now fails
import joblib, numpy as np
d = joblib.load("{BUNDLE_PATH}")
print("loaded WITHOUT pandas/sklearn. keys:", sorted(d))
print("shapes:", d["X_train"].shape, d["X_val"].shape, d["X_test"].shape,
      "| dtypes:", d["X_train"].dtype, d["y_train"].dtype, d["index_train"].dtype)
print("target_transform:", d["target_transform"], "| reference_id:", d["fingerprint"]["reference"]["reference_id"])
'''
r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
print(r.stdout or r.stderr)
assert r.returncode == 0, "bundle must load in a fresh process"

loaded WITHOUT pandas/sklearn. keys: ['X_test', 'X_train', 'X_val', 'feature_names', 'fingerprint', 'index_test', 'index_train', 'index_val', 'meta', 'preprocessor', 'preprocessor_format', 'storage_format', 'target_transform', 'y_test', 'y_train', 'y_val']
shapes: (3313, 34) (711, 34) (711, 34) | dtypes: float64 int64 int64
target_transform: log1p | reference_id: 70560d1bd70af7cd



## 8. Summary of the frozen data

In [13]:
summary = pd.DataFrame({
    "rows": [len(X_train), len(X_val), len(X_test)],
    "share_%": np.round(np.array([len(X_train), len(X_val), len(X_test)]) / (len(X_train) + len(X_val) + len(X_test)) * 100, 1),
    "rent_median_₹": [y_train.median(), y_val.median(), y_test.median()],
    "rent_mean_₹": [round(y_train.mean()), round(y_val.mean()), round(y_test.mean())],
    "rent_max_₹": [y_train.max(), y_val.max(), y_test.max()],
}, index=["train", "val", "test"])
display(summary)
print(f"Features: {len(feature_names)} | target_transform: {TARGET_TRANSFORM} | reference_id: {EXPECTED_REFERENCE_ID}")

,rows,share_%,rent_median_₹,rent_mean_₹,rent_max_₹
train,3313,70.0,16000.0,34169,1000000
val,711,15.0,16000.0,35596,1200000
test,711,15.0,16000.0,33126,700000


Features: 34 | target_transform: log1p | reference_id: 70560d1bd70af7cd



## 9. Phase-5 record (frozen)

**Preparation is now frozen.** Every later notebook loads `prepared_data.joblib` and does no cleaning.

| Item | Value |
|---|---|
| Pipeline order | rules (4,746 → 4,737) → stateless features → q=10 stratified 70/15/15 split (seed 42) → train-only far-out filter (−2 train rows) → `ColumnTransformer` fit on X_train with `log1p(y_train)` → transform all splits |
| Rows | train **3,313** · val **711** · test **711** (val/test unfiltered) |
| Features | **34**, float64, unscaled (scaling belongs to the model notebooks) |
| y | int64 **rupees**; `target_transform = "log1p"` → wrap models in `TransformedTargetRegressor(func=np.log1p, inverse_func=np.expm1)` |
| **Storage** | **numpy arrays + Python builtins only** (`X_*` float64 2-D, `y_*` int64, `index_*` int64 row ids, `feature_names` list). There are no pandas or sklearn objects, so it loads under any pandas version, including Colab's pandas 2.x. Rebuild frames with `frames_from_bundle(d)`. |
| Split check | val/test sorted-index hashes equal Phase 3 (`55e1909c244731ec`, `1874e7d6b55e66dd`). The train hash changed only because of the 2 documented far-out drops. |
| **`EXPECTED_REFERENCE_ID`** | **`70560d1bd70af7cd`**. Model notebooks hard-code this and STOP on mismatch. |
| Content hashes | train `45607f3d45155c30` · val `9abca1d0c8a63c34` · test `ccff150b1ed3fc71` |
| Reproducibility | Two independent runs gave an identical fingerprint. |
| Portability | The bundle loads in a fresh process where **pandas and sklearn cannot be imported**. The cloudpickled preprocessor (reference only) restores and reproduces X_val/X_test exactly under the versions in `meta.versions`. |

**Revision note:** the first version of this bundle stored pandas DataFrames. pandas 3 pickles string column labels with a dtype that pandas 2.x cannot read, so Colab failed with `TypeError: StringDtype.__init__()`. The storage format was changed to numpy arrays. The data, split, features and fingerprint are unchanged (same reference ID and content hashes).

**Accepted design notes (signed off after Phase 4):**
1. CV on the saved matrices does not re-fit the locality encoder per fold. Phase 4 measured this at ≤0.5% RMSE.
2. `preprocessor` is stored as cloudpickle bytes.

**Model-notebook contract (Phase 6+):** `joblib.load("prepared_data.joblib")` → `frames_from_bundle(d)` → verify the fingerprint against `"70560d1bd70af7cd"` →
`Pipeline([scaler if linear, model])` inside `TransformedTargetRegressor(log1p/expm1)` → fit on X_train → CV RMSE on train
(5-fold, seed 42) → validation metrics in ₹ → test only when `EVALUATE_TEST = True` (Phase 9).
